## Population Preprocessing

This notebook demonstrates the population-data preprocessing workflow used in the project, including administrative-area harmonization, population feature construction, data integration, and validation.

### 1. Imports & Configuration


In [ ]:
from __future__ import annotations

import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

# Update these paths for your local environment.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")

POPULATION_WORKBOOK = DATA_DIR / "population_by_age.xlsx"
ALL_INPUT = DATA_DIR / "all_analysis_data.xlsx"
GENDER_INPUT = DATA_DIR / "gender_analysis_data.xlsx"

ALL_OUTPUT = OUTPUT_DIR / "all_with_population_55_84.xlsx"
GENDER_OUTPUT = OUTPUT_DIR / "gender_with_population_55_84.xlsx"
UNMATCHED_OUTPUT = OUTPUT_DIR / "unmatched_population_records.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AGE_MIN = 55
AGE_MAX = 84

# Column positions in the original population workbook.
# pandas iloc uses a half-open interval [start, end).
AGE_START_COL = 58
AGE_END_COL = 88


# ============================================================
# Text and administrative-area normalization
# ============================================================

def normalize_text(value) -> str:
    """Normalize text for consistent matching."""
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = (
        text.replace("\ufeff", "")
        .replace("　", "")
        .replace(" ", "")
        .strip()
        .replace("臺", "台")
    )

    # Known source-text corrections observed in the original files.
    text = text.replace("捕里", "埔里").replace("蘆賽", "褒忠")
    return text


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize dataframe column names."""
    result = df.copy()
    result.columns = [normalize_text(col) for col in result.columns]
    return result


def to_roc_year(series: pd.Series) -> pd.Series:
    """Convert Gregorian years to ROC years; keep ROC years unchanged."""
    year = pd.to_numeric(series, errors="coerce")
    converted = np.where(year > 1900, year - 1911, year)
    return pd.Series(converted, index=series.index, dtype="Int64")


def strip_admin_suffix(name: str) -> str:
    """
    Create a fallback township key by removing one trailing
    administrative suffix: 市 / 鎮 / 鄉 / 區.
    """
    return re.sub(r"[市鎮鄉區]$", "", normalize_text(name))


CITY_NAME_MAP = {
    "台北縣": "新北市",
    "台中縣": "台中市",
    "台南縣": "台南市",
    "高雄縣": "高雄市",
    "桃園縣": "桃園市",
    "原台北縣": "新北市",
    "原台中市": "台中市",
    "原台中縣": "台中市",
    "原台南市": "台南市",
    "原台南縣": "台南市",
    "原高雄市": "高雄市",
    "原高雄縣": "高雄市",
}


### 2. Administrative-Area Standardization


In [ ]:
def normalize_city(value) -> str:
    """Harmonize historical county/city names."""
    city = normalize_text(value)
    if city.startswith("原"):
        city = city[1:]
    return CITY_NAME_MAP.get(city, city)


TAOYUAN_TOWNSHIP_MAP = {
    "桃園市": "桃園區",
    "中壢市": "中壢區",
    "平鎮市": "平鎮區",
    "八德市": "八德區",
    "大溪鎮": "大溪區",
    "楊梅市": "楊梅區",
    "龍潭鄉": "龍潭區",
    "龜山鄉": "龜山區",
    "蘆竹鄉": "蘆竹區",
    "大園鄉": "大園區",
    "觀音鄉": "觀音區",
    "新屋鄉": "新屋區",
    "復興鄉": "復興區",
}


def normalize_township(city: str, township: str) -> str:
    """
    Harmonize township names.

    Taoyuan requires additional name harmonization because the historical
    township/city names changed after the administrative reorganization.
    """
    city = normalize_city(city)
    township = normalize_text(township)

    if city == "桃園市":
        return TAOYUAN_TOWNSHIP_MAP.get(township, township)

    return township


# ============================================================
# Population workbook parsing
# ============================================================

COUNTY_NAMES_RAW = {
    "台北市", "新北市", "台中市", "台南市", "高雄市", "桃園市", "桃園縣",
    "基隆市", "新竹市", "嘉義市", "新竹縣", "苗栗縣", "彰化縣",
    "南投縣", "雲林縣", "嘉義縣", "屏東縣", "宜蘭縣", "花蓮縣",
    "台東縣", "澎湖縣", "金門縣", "連江縣",
}

COUNTY_NAMES_RAW_N = sorted(
    {normalize_text(name) for name in COUNTY_NAMES_RAW},
    key=len,
    reverse=True,
)

COUNTY_NAMES_N = sorted(
    {normalize_city(name) for name in COUNTY_NAMES_RAW},
    key=len,
    reverse=True,
)

SKIP_TOKENS = {
    normalize_text(value)
    for value in {"台灣地區", "福建省", "總計"}
}


def split_city_township(area_raw: str) -> tuple[str | None, str | None]:
    """
    Split an area label into city/county and township when both appear
    in the same cell.

    Examples
    --------
    桃園縣       -> ("桃園市", "")
    桃園縣中壢市 -> ("桃園市", "中壢市")
    彰化縣員林市 -> ("彰化縣", "員林市")
    """
    area = normalize_text(area_raw)
    if not area:
        return None, None

    # Check historical/raw county names first so that old names such as
    # 桃園縣 are identified before being normalized to 桃園市.
    for raw_city in COUNTY_NAMES_RAW_N:
        if area == raw_city:
            return normalize_city(raw_city), ""
        if area.startswith(raw_city) and len(area) > len(raw_city):
            return normalize_city(raw_city), area[len(raw_city):]

    for city in COUNTY_NAMES_N:
        if area == city:
            return city, ""
        if area.startswith(city) and len(area) > len(city):
            return city, area[len(city):]

    return None, None


### 3. Population Feature Construction


In [ ]:
def find_population_start_row(df_raw: pd.DataFrame) -> int:
    """Locate the first population row containing sex and area information."""
    for row_idx in range(len(df_raw)):
        sex_text = (
            normalize_text(df_raw.iat[row_idx, 0])
            if df_raw.shape[1] > 0
            else ""
        )
        area_text = (
            normalize_text(df_raw.iat[row_idx, 1])
            if df_raw.shape[1] > 1
            else ""
        )

        if (
            ("男" in sex_text or "女" in sex_text)
            and area_text
            and "nan" not in area_text.lower()
        ):
            return row_idx

    # Fallback retained from the original workbook structure.
    return 7


def build_population_database(
    workbook_path: Path,
    roc_years,
    age_start_col: int = AGE_START_COL,
    age_end_col: int = AGE_END_COL,
) -> pd.DataFrame:
    """
    Parse the population workbook and construct sex-specific population
    counts and weighted age sums for ages 55-84.
    """
    if not workbook_path.exists():
        raise FileNotFoundError(f"Population workbook not found: {workbook_path}")

    excel_file = pd.ExcelFile(workbook_path)
    all_rows: list[dict] = []
    age_vector = np.arange(AGE_MIN, AGE_MAX + 1)

    expected_age_columns = AGE_MAX - AGE_MIN + 1
    if age_end_col - age_start_col != expected_age_columns:
        raise ValueError(
            "Age-column range does not match the expected number of ages "
            f"({expected_age_columns})."
        )

    for year in sorted({int(y) for y in roc_years if pd.notna(y)}):
        sheet_name = str(year)
        if sheet_name not in excel_file.sheet_names:
            continue

        df = excel_file.parse(sheet_name, header=None)
        start_row = find_population_start_row(df)

        current_city = ""
        current_gender = None

        for row_idx in range(start_row, len(df)):
            gender_raw = (
                normalize_text(df.iat[row_idx, 0])
                if df.shape[1] > 0
                else ""
            )
            area_raw = (
                normalize_text(df.iat[row_idx, 1])
                if df.shape[1] > 1
                else ""
            )

            if not area_raw or "nan" in area_raw.lower():
                continue
            if area_raw in SKIP_TOKENS or "總計" in area_raw:
                continue

            gender = (
                "M" if "男" in gender_raw
                else "F" if "女" in gender_raw
                else ""
            )
            if not gender:
                continue

            # Reset the city context when the workbook switches sex sections.
            if current_gender is None:
                current_gender = gender
            elif current_gender != gender:
                current_gender = gender
                current_city = ""

            city, township = split_city_township(area_raw)

            if city is not None:
                # A city/county-only row updates the current city context.
                if township == "":
                    current_city = city
                    continue

                use_city = city
                use_township = township

            else:
                # Traditional workbook layout: city/county appears on a
                # separate row followed by township rows.
                if (
                    area_raw in COUNTY_NAMES_RAW_N
                    or normalize_city(area_raw) in COUNTY_NAMES_N
                ):
                    current_city = area_raw
                    continue

                if not current_city:
                    continue

                use_city = current_city
                use_township = area_raw

            use_city = normalize_city(use_city)
            use_township = normalize_township(use_city, use_township)

            age_values = pd.to_numeric(
                df.iloc[row_idx, age_start_col:age_end_col],
                errors="coerce",
            ).fillna(0).to_numpy(dtype=float)

            if len(age_values) != len(age_vector):
                raise ValueError(
                    f"Unexpected age-column length in year {year}, row {row_idx}: "
                    f"{len(age_values)} values found."
                )

            population_sum = int(np.rint(age_values.sum()))
            weighted_age_sum = float(np.sum(age_values * age_vector))

            all_rows.append(
                {
                    "roc_year": year,
                    "縣市": use_city,
                    "鄉鎮市區": use_township,
                    "gender": gender,
                    "population_55_84": population_sum,
                    "weighted_age_sum": weighted_age_sum,
                }
            )

    population_db = pd.DataFrame(all_rows)

    if population_db.empty:
        raise ValueError(
            "No population records were parsed. Check sheet names, "
            "start rows, and age-column positions."
        )

    population_db["縣市"] = population_db["縣市"].apply(normalize_city)
    population_db["鄉鎮市區"] = population_db["鄉鎮市區"].apply(normalize_text)
    population_db["town_key"] = population_db["鄉鎮市區"].apply(strip_admin_suffix)

    return population_db


### 4. Data Integration


In [ ]:
def build_population_metrics(population_db: pd.DataFrame) -> pd.DataFrame:
    """Aggregate sex-specific records into township-level population metrics."""
    pivot = population_db.pivot_table(
        index=["roc_year", "縣市", "鄉鎮市區", "town_key"],
        columns="gender",
        values=["population_55_84", "weighted_age_sum"],
        aggfunc="sum",
    ).fillna(0)

    pivot.columns = [f"{metric}_{gender}" for metric, gender in pivot.columns]
    pivot = pivot.reset_index()

    male_population = pivot.get(
        "population_55_84_M",
        pd.Series(0, index=pivot.index),
    )
    female_population = pivot.get(
        "population_55_84_F",
        pd.Series(0, index=pivot.index),
    )
    male_weighted_age = pivot.get(
        "weighted_age_sum_M",
        pd.Series(0, index=pivot.index),
    )
    female_weighted_age = pivot.get(
        "weighted_age_sum_F",
        pd.Series(0, index=pivot.index),
    )

    pivot["男性人口數_55_84"] = male_population.astype(int)
    pivot["女性人口數_55_84"] = female_population.astype(int)
    pivot["55_84_總人口數"] = (
        pivot["男性人口數_55_84"] + pivot["女性人口數_55_84"]
    )

    denominator = pivot["55_84_總人口數"].replace(0, np.nan)

    pivot["平均年齡_55_84"] = (
        male_weighted_age + female_weighted_age
    ) / denominator

    pivot["男性比例_55_84"] = (
        pivot["男性人口數_55_84"] / denominator
    )

    output_columns = [
        "roc_year",
        "縣市",
        "鄉鎮市區",
        "town_key",
        "男性人口數_55_84",
        "女性人口數_55_84",
        "55_84_總人口數",
        "平均年齡_55_84",
        "男性比例_55_84",
    ]

    return pivot[output_columns].drop_duplicates()


# ============================================================
# Population merge and validation
# ============================================================

POPULATION_COLUMNS = [
    "男性人口數_55_84",
    "女性人口數_55_84",
    "55_84_總人口數",
    "平均年齡_55_84",
    "男性比例_55_84",
]


def merge_population_metrics(
    df: pd.DataFrame,
    population_metrics: pd.DataFrame,
    dataset_type: str,
    unmatched_records: list[pd.DataFrame],
    sheet_name: str | None = None,
) -> pd.DataFrame:
    """
    Merge population metrics in two stages.

    Stage 1
        Exact match on year, city/county, and township.

    Stage 2
        For unmatched rows, use a fallback township key that removes the
        final administrative suffix. The fallback is used only when the key
        is unique within a year and city/county.
    """
    result = df.copy()

    required_columns = {"年度", "縣市", "鄉鎮市區"}
    missing_columns = required_columns - set(result.columns)
    if missing_columns:
        raise KeyError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    result["roc_year"] = to_roc_year(result["年度"])
    result["縣市"] = result["縣市"].apply(normalize_city)
    result["鄉鎮市區"] = result["鄉鎮市區"].apply(normalize_text)
    result["town_key"] = result["鄉鎮市區"].apply(strip_admin_suffix)

    # Stage 1: exact administrative-area match.
    exact_lookup = population_metrics[
        ["roc_year", "縣市", "鄉鎮市區", *POPULATION_COLUMNS]
    ].drop_duplicates(
        subset=["roc_year", "縣市", "鄉鎮市區"],
        keep="first",
    )

    merged = result.merge(
        exact_lookup,
        on=["roc_year", "縣市", "鄉鎮市區"],
        how="left",
        validate="m:1",
    )

    missing_mask = merged["55_84_總人口數"].isna()

    # Stage 2: fallback match using suffix-stripped township keys.
    if missing_mask.any():
        fallback_source = population_metrics[
            ["roc_year", "縣市", "town_key", *POPULATION_COLUMNS]
        ].copy()

        # Keep fallback keys only when they map to a single population row.
        key_counts = (
            fallback_source
            .groupby(["roc_year", "縣市", "town_key"], dropna=False)
            .size()
            .rename("key_count")
            .reset_index()
        )

        unique_keys = key_counts.loc[
            key_counts["key_count"] == 1,
            ["roc_year", "縣市", "town_key"],
        ]

        fallback_lookup = (
            fallback_source
            .merge(
                unique_keys,
                on=["roc_year", "縣市", "town_key"],
                how="inner",
            )
            .drop_duplicates(
                subset=["roc_year", "縣市", "town_key"],
                keep="first",
            )
        )

        rows_to_fill = merged.loc[
            missing_mask,
            ["roc_year", "縣市", "town_key"],
        ].copy()
        rows_to_fill["_row_id"] = rows_to_fill.index

        fallback_matches = rows_to_fill.merge(
            fallback_lookup,
            on=["roc_year", "縣市", "town_key"],
            how="left",
            validate="m:1",
        ).set_index("_row_id")

        for column in POPULATION_COLUMNS:
            merged.loc[
                fallback_matches.index,
                column,
            ] = fallback_matches[column]

    # Collect records that still cannot be matched.
    unmatched_mask = merged["55_84_總人口數"].isna()

    if unmatched_mask.any():
        columns_to_export = [
            col
            for col in ["roc_year", "年度", "縣市", "鄉鎮市區"]
            if col in merged.columns
        ]

        unmatched = merged.loc[
            unmatched_mask,
            columns_to_export,
        ].copy()

        unmatched["dataset_type"] = dataset_type

        if sheet_name is not None:
            unmatched["sheet"] = sheet_name

        unmatched_records.append(unmatched)

    # Population counts remain nullable here so that unmatched rows are
    # distinguishable from true zero-population values.
    for column in [
        "男性人口數_55_84",
        "女性人口數_55_84",
        "55_84_總人口數",
    ]:
        merged[column] = pd.to_numeric(
            merged[column],
            errors="coerce",
        ).astype("Int64")

    return merged


# ============================================================
# Main workflow
# ============================================================


### 5. Validation & Export


In [ ]:
def get_available_roc_years(workbook_path: Path) -> list[int]:
    """Return numeric ROC-year sheet names available in the workbook."""
    excel_file = pd.ExcelFile(workbook_path)

    return sorted(
        int(sheet)
        for sheet in excel_file.sheet_names
        if re.fullmatch(r"\d{2,3}", str(sheet))
    )


def main() -> None:
    if not ALL_INPUT.exists():
        raise FileNotFoundError(f"Analysis file not found: {ALL_INPUT}")

    if not GENDER_INPUT.exists():
        raise FileNotFoundError(f"Gender file not found: {GENDER_INPUT}")

    # --------------------------------------------------------
    # Determine study years and build population metrics
    # --------------------------------------------------------
    all_data = clean_column_names(pd.read_excel(ALL_INPUT))

    if "年度" not in all_data.columns:
        raise KeyError("Column '年度' was not found in ALL_INPUT.")

    all_data["roc_year"] = to_roc_year(all_data["年度"])

    available_years = get_available_roc_years(POPULATION_WORKBOOK)
    required_years = sorted(
        {int(year) for year in all_data["roc_year"].dropna().unique()}
    )

    years_to_load = sorted(
        set(required_years).intersection(available_years)
    )

    if not years_to_load:
        raise ValueError(
            "No overlapping years were found between the analysis data "
            "and the population workbook."
        )

    print(
        f"Analysis years: {min(required_years)}-{max(required_years)}"
    )
    print(
        f"Population workbook years: "
        f"{min(available_years)}-{max(available_years)}"
    )
    print(
        f"Years used for population preprocessing: "
        f"{min(years_to_load)}-{max(years_to_load)}"
    )

    population_db = build_population_database(
        POPULATION_WORKBOOK,
        roc_years=years_to_load,
    )

    population_metrics = build_population_metrics(population_db)

    unmatched_records: list[pd.DataFrame] = []

    # --------------------------------------------------------
    # Merge population metrics into the main analysis dataset
    # --------------------------------------------------------
    merged_all = merge_population_metrics(
        all_data,
        population_metrics,
        dataset_type="ALL",
        unmatched_records=unmatched_records,
    )

    matched_all = merged_all["55_84_總人口數"].notna()
    print(
        f"[ALL] Population match rate: {matched_all.mean():.2%}"
    )

    merged_all = merged_all.drop(
        columns=["roc_year", "town_key"],
        errors="ignore",
    )
    merged_all.to_excel(ALL_OUTPUT, index=False)

    # --------------------------------------------------------
    # Merge population metrics into sex-specific worksheets
    # --------------------------------------------------------
    gender_book = pd.ExcelFile(GENDER_INPUT)

    with pd.ExcelWriter(GENDER_OUTPUT, engine="openpyxl") as writer:
        for sheet_name in gender_book.sheet_names:
            gender_data = clean_column_names(
                gender_book.parse(sheet_name)
            )

            try:
                merged_gender = merge_population_metrics(
                    gender_data,
                    population_metrics,
                    dataset_type="GENDER",
                    unmatched_records=unmatched_records,
                    sheet_name=sheet_name,
                )

                if "性別" in merged_gender.columns:
                    sex_code = merged_gender["性別"].map(
                        {
                            "M": "M",
                            "男": "M",
                            "F": "F",
                            "女": "F",
                        }
                    )

                    merged_gender["該性別55_84人口數"] = np.where(
                        sex_code == "M",
                        merged_gender["男性人口數_55_84"],
                        np.where(
                            sex_code == "F",
                            merged_gender["女性人口數_55_84"],
                            pd.NA,
                        ),
                    )

                    merged_gender["該性別55_84人口數"] = pd.to_numeric(
                        merged_gender["該性別55_84人口數"],
                        errors="coerce",
                    ).astype("Int64")

                merged_gender = merged_gender.drop(
                    columns=["roc_year", "town_key"],
                    errors="ignore",
                )

                merged_gender.to_excel(
                    writer,
                    sheet_name=sheet_name[:31],
                    index=False,
                )

            except Exception as exc:
                error_sheet = f"_ERROR_{sheet_name}"[:31]
                pd.DataFrame(
                    {
                        "sheet": [sheet_name],
                        "error": [repr(exc)],
                    }
                ).to_excel(
                    writer,
                    sheet_name=error_sheet,
                    index=False,
                )

    # --------------------------------------------------------
    # Export unmatched records for manual validation
    # --------------------------------------------------------
    if unmatched_records:
        unmatched_df = (
            pd.concat(unmatched_records, ignore_index=True)
            .drop_duplicates()
        )
    else:
        unmatched_df = pd.DataFrame(
            columns=[
                "roc_year",
                "年度",
                "縣市",
                "鄉鎮市區",
                "dataset_type",
                "sheet",
            ]
        )

    unmatched_df.to_excel(UNMATCHED_OUTPUT, index=False)

    print(f"Saved: {ALL_OUTPUT}")
    print(f"Saved: {GENDER_OUTPUT}")
    print(f"Saved: {UNMATCHED_OUTPUT}")
    print("Population preprocessing completed.")


if __name__ == "__main__":
    main()
